# Week 9 - Day 2: Tools + Function Calling

## Learning Objectives

By the end of this notebook, you will understand:

1. What an AI Agent tool is
2. What a Python function is
3. How a function can become an agent tool
4. What function calling means
5. Tool descriptions
6. Tool parameters
7. Tool schemas
8. Tool selection
9. Tool arguments
10. Tool execution
11. Tool results
12. The tool-calling loop
13. Multiple tools
14. Error handling
15. Tool validation
16. Tool permissions
17. How tools connect to AI Agents

## Core Flow
```
User
 ↓
LLM / Agent
 ↓
Choose Tool
 ↓
Generate Arguments
 ↓
Application Executes Tool
 ↓
Tool Result
 ↓
LLM / Agent
 ↓
Final Answer

# 1. What Is an Agent Tool?

A tool is an external function or capability that an AI Agent
can use to perform an action.

Examples:

- Calculator
- Web search
- Database query
- File reader
- Python execution
- Weather API
- Email service
- Calendar API
- CRM system
- RAG search

An LLM can decide that a tool is needed.

The application is responsible for actually executing the tool.

# Example

User:

"What is 250 × 40?"

The agent may decide:

I need a calculator.

The agent requests:

calculator(
    250,
    40,
    "multiply"
)

The application executes the function.

The tool returns:

10000

The LLM then generates:

"250 × 40 = 10,000."

The important separation is:

LLM → decides

Application → executes

Tool → returns result

# 2. Python Functions

Before understanding function calling, we need to understand
ordinary Python functions.

A function:

- Receives inputs
- Performs an operation
- Returns an output

Example:

calculate_total(price, quantity)

Input:
price = 100
quantity = 5

Output:
500

In [1]:
def calculate_total(price, quantity):
    return price * quantity

In [2]:
result = calculate_total(100, 5)

print("Total:", result)

Total: 500


# 3. Function → Tool

A normal Python function is written for a programmer.

An AI tool needs additional information so that an LLM can
understand how to use it.

For example:

Function:

calculate_total(price, quantity)

Tool description:

"Calculate the total price by multiplying price and quantity."

Parameters:

price → number
quantity → number

The LLM needs this information to select and call the tool correctly.

In [3]:
def calculate_total(price, quantity):
    return price * quantity


tool_definition = {
    "name": "calculate_total",
    "description": "Calculate the total price using price and quantity.",
    "parameters": {
        "price": "number",
        "quantity": "number"
    }
}

print(tool_definition)

{'name': 'calculate_total', 'description': 'Calculate the total price using price and quantity.', 'parameters': {'price': 'number', 'quantity': 'number'}}


# 4. What Is a Tool Schema?

A tool schema describes how a tool should be called.

It usually contains:

- Tool name
- Description
- Parameters
- Parameter types
- Required parameters

Example:

{
    "name": "calculate_total",
    "description": "Calculate total price",
    "parameters": {
        "price": number,
        "quantity": number
    }
}

The schema acts like a contract between:

LLM
and
Application

# Why Is a Schema Important?

Suppose the function expects:

price
quantity

But the model sends:

product_name
color
size

The application cannot correctly execute the function.

A clear schema tells the model exactly what arguments are required.

Therefore:

Good tool schema
→ Better tool selection
→ Better arguments
→ More reliable execution

# 5. Tool Parameters

A tool can have:

- No parameters
- One parameter
- Multiple parameters
- Required parameters
- Optional parameters

Example 1:

get_current_time()

No parameters.

Example 2:

get_weather(city)

One parameter.

Example 3:

calculate_total(price, quantity)

Two parameters.

Example 4:

search_products(
    query,
    category,
    max_results
)

Multiple parameters.

In [4]:
def get_weather(city):
    return f"Weather information for {city}"


def calculate_discount(price, discount_percent):
    discount = price * discount_percent / 100
    return price - discount

In [5]:
print(get_weather("Chennai"))

print(
    calculate_discount(
        1000,
        10
    )
)

Weather information for Chennai
900.0


# 6. Required vs Optional Parameters

Some parameters must be provided.

Example:

calculate_total(price, quantity)

Both are required.

Other parameters can have defaults.

Example:

search_products(
    query,
    max_results=5
)

Here:

query → required

max_results → optional

Clear parameter definitions make tool calling more reliable.

In [6]:
def search_products(query, max_results=5):
    return {
        "query": query,
        "max_results": max_results
    }


print(search_products("laptops"))

print(
    search_products(
        "laptops",
        max_results=10
    )
)

{'query': 'laptops', 'max_results': 5}
{'query': 'laptops', 'max_results': 10}


# 7. What Is Function Calling?

Function calling allows an LLM to request that the application
execute a specific function.

The LLM does not directly execute the Python function.

Instead, it produces a structured tool-call request.

Conceptual flow:

User
 ↓
LLM
 ↓
Tool Call Request
 ↓
Application
 ↓
Python Function
 ↓
Tool Result
 ↓
LLM
 ↓
Final Response

# Example

User:

"Calculate the total cost of 5 products costing ₹250 each."

The LLM may produce a structured request:

Tool:
calculate_total

Arguments:

{
    "price": 250,
    "quantity": 5
}

The application receives this request.

It executes:

calculate_total(250, 5)

Result:

1250

The result is returned to the LLM.

The LLM produces:

"The total cost is ₹1,250."

# 8. LLM vs Tool Execution

This is one of the most important concepts of today's lesson.

### LLM

Responsible for:

- Understanding the user's request
- Deciding whether a tool is needed
- Selecting the tool
- Generating tool arguments
- Interpreting tool results
- Producing the final response

### Application

Responsible for:

- Receiving tool calls
- Validating arguments
- Executing functions
- Handling errors
- Returning tool results

### Tool

Responsible for:

- Performing the actual operation
- Returning a result

# 9. Simulating Function Calling

Before using an actual LLM API, we can simulate the process.

Suppose the model returns:

{
    "tool": "calculate_total",
    "arguments": {
        "price": 250,
        "quantity": 5
    }
}

Our application needs to:

1. Read the tool name
2. Read the arguments
3. Find the corresponding Python function
4. Execute it
5. Return the result

In [7]:
def calculate_total(price, quantity):
    return price * quantity


tool_call = {
    "tool": "calculate_total",
    "arguments": {
        "price": 250,
        "quantity": 5
    }
}

print("Tool:", tool_call["tool"])
print("Arguments:", tool_call["arguments"])

Tool: calculate_total
Arguments: {'price': 250, 'quantity': 5}


In [8]:
result = calculate_total(
    tool_call["arguments"]["price"],
    tool_call["arguments"]["quantity"]
)

print("Tool result:", result)

Tool result: 1250


# 10. Tool Registry

An agent may have many tools.

Instead of manually checking every function, we can create
a tool registry.

Example:

{
    "calculator": calculator_function,
    "weather": weather_function,
    "search": search_function
}

The agent can use the tool name to find the corresponding function.

In [9]:
def calculator(a, b):
    return a + b


def get_weather(city):
    return f"Weather information for {city}"


def search_database(query):
    return f"Database results for: {query}"


tool_registry = {
    "calculator": calculator,
    "get_weather": get_weather,
    "search_database": search_database
}

print(tool_registry.keys())

dict_keys(['calculator', 'get_weather', 'search_database'])


In [10]:
tool_name = "calculator"

tool = tool_registry[tool_name]

result = tool(10, 20)

print("Result:", result)

Result: 30


# Why a Tool Registry Helps

Suppose we have:

10 tools

Instead of writing:

if tool == "calculator":
    ...

elif tool == "weather":
    ...

elif tool == "database":
    ...

We can maintain a registry.

Tool name
↓
Tool function

This makes the application easier to extend.

# 11. Multiple Tools

A useful agent usually has access to multiple tools.

Example:
```
                 AGENT
                   |
       ┌───────────┼───────────┐
       ↓           ↓           ↓
 Calculator     Search      Database
       |           |           |
       └───────────┼───────────┘
                   ↓
                Result

Possible user requests:

"Calculate 25 × 50"
→ Calculator

"Find recent AI research"
→ Search

"Find employee E102"
→ Database

The agent needs to choose the appropriate tool.

# 12. Tool Selection

Tool selection means deciding which available tool should be used.

Example:

User:
"What is 50 × 20?"

Available tools:

- Calculator
- Search
- Database
- File Reader

Best tool:

Calculator

The model should select a tool based on:

- User intent
- Tool description
- Required parameters
- Available capabilities

In [11]:
tools = [
    {
        "name": "calculator",
        "description": "Performs mathematical calculations."
    },
    {
        "name": "search",
        "description": "Searches for information on the web."
    },
    {
        "name": "database",
        "description": "Retrieves information from a database."
    }
]

for tool in tools:
    print(tool["name"], "→", tool["description"])

calculator → Performs mathematical calculations.
search → Searches for information on the web.
database → Retrieves information from a database.


# 13. Tool Arguments

Selecting the correct tool is only half of the problem.

The agent must also provide correct arguments.

Example:

Tool:

get_weather(city)

User:

"What is the weather in Chennai?"

Correct tool call:

get_weather(
    city="Chennai"
)

Incorrect:

get_weather(
    city="Mumbai"
)

The application should therefore validate tool arguments before execution.

# 14. Argument Validation

Tool arguments should be validated before execution.

Example:

calculate_discount(
    price=1000,
    discount_percent=10
)

Valid.

But:

calculate_discount(
    price="hello",
    discount_percent="ten"
)

Invalid.

Validation protects the application from malformed tool calls.

In [12]:
def calculate_discount(price, discount_percent):
    if not isinstance(price, (int, float)):
        raise ValueError("price must be a number")

    if not isinstance(discount_percent, (int, float)):
        raise ValueError("discount_percent must be a number")

    if discount_percent < 0 or discount_percent > 100:
        raise ValueError(
            "discount_percent must be between 0 and 100"
        )

    discount = price * discount_percent / 100

    return price - discount

In [13]:
print(
    calculate_discount(
        1000,
        20
    )
)

800.0


In [14]:
try:
    result = calculate_discount(
        "1000",
        20
    )

    print(result)

except ValueError as error:
    print("Tool error:", error)

Tool error: price must be a number


# 15. Tool Errors

Tools can fail.

Examples:

- API unavailable
- Database connection failure
- Invalid arguments
- File not found
- Permission denied
- Network timeout
- Rate limit
- Invalid response

A robust agent should not simply crash.

Instead:

Tool
 ↓
Error
 ↓
Application catches error
 ↓
Return structured error
 ↓
Agent decides what to do next

In [15]:
def safe_calculator(a, b, operation):
    try:
        if operation == "add":
            return {
                "success": True,
                "result": a + b
            }

        elif operation == "multiply":
            return {
                "success": True,
                "result": a * b
            }

        else:
            return {
                "success": False,
                "error": "Unsupported operation"
            }

    except Exception as error:
        return {
            "success": False,
            "error": str(error)
        }

In [16]:
result = safe_calculator(
    10,
    5,
    "multiply"
)

print(result)

{'success': True, 'result': 50}


# 16. Structured Tool Results

Instead of returning random text, tools can return structured data.

Example:

{
    "success": true,
    "result": 50
}

Or:

{
    "success": false,
    "error": "Database connection failed"
}

Structured results make it easier for the agent to understand
what happened.

# 17. Complete Function Calling Loop

The process can be represented as:
```
USER
 ↓
LLM
 ↓
Does the task require a tool?
 ↓
YES
 ↓
Select Tool
 ↓
Generate Arguments
 ↓
Application Validates Arguments
 ↓
Execute Tool
 ↓
Tool Result
 ↓
Send Result Back to LLM
 ↓
LLM Interprets Result
 ↓
Task Complete?
 ↙          ↘
NO          YES
↓             ↓
Another      Final
Tool Call    Answer

# 18. Multiple Tool Calls

Some tasks require one tool.

Example:

"Calculate 20 × 50."

One tool call:

Calculator
→ 1000

Other tasks require multiple tools.

Example:

"Find the total sales from sales.xlsx and summarize the result."

Possible sequence:

1. File Reader
2. Data Analysis
3. Calculator
4. Writer / LLM

Therefore, an agent may need to perform multiple tool calls
before producing the final answer.

# 19. Sequential Tool Calling

Some tool calls depend on earlier results.

Example:

Step 1:
Read sales file.

Result:
File contains sales data.

Step 2:
Calculate total sales.

Result:
₹50,00,000.

Step 3:
Calculate average order value.

Result:
₹1,250.

Step 4:
Generate report.

This is sequential tool execution.

The result of one action becomes information for the next action.

# 20. Parallel Tool Calls

Some independent tasks can potentially be performed in parallel.

Example:

User:
"Get the weather in Chennai and Bengaluru."

The two requests are independent.

Conceptually:

              AGENT
              /   \
             ↓     ↓
        Chennai   Bengaluru
         Weather    Weather
             \     /
              ↓   ↓
              RESULT

Parallel execution can reduce latency when tools support it.

However, parallel execution should only be used when the
operations are independent and safe to run concurrently.

# 21. Tool Permissions

Not every agent should have access to every tool.

Example:

Research Agent:
- Web Search
- Document Search

Data Agent:
- Python
- Database Read

Writer Agent:
- Document Generation

Finance Agent:
- Financial Data

An agent should only receive the tools it actually needs.

This follows the principle:

Least privilege

Give an agent only the permissions necessary for its task.

# 22. Read Tools vs Write Tools

Tools can have different levels of risk.

## Read Tools

Examples:

- Search
- Read file
- Query database
- Retrieve document

These generally read information.

## Write Tools

Examples:

- Send email
- Delete file
- Update database
- Create order
- Change account settings

Write operations can cause external side effects.

For sensitive operations, the system may require:

Agent
↓
Request approval
↓
User confirmation
↓
Tool execution

# 23. Tool Security

Function calling introduces security considerations.

Important protections include:

### 1. Input validation

Validate all tool arguments.

### 2. Authentication

Verify that the application has permission to perform the action.

### 3. Authorization

Check whether the current user is allowed to perform it.

### 4. Least privilege

Give agents only the tools they need.

### 5. Logging

Record important tool calls.

### 6. Rate limiting

Prevent excessive tool execution.

### 7. Approval

Require user approval for sensitive operations.

### 8. Timeouts

Prevent tools from running indefinitely.

In [17]:
tool_log = []

tool_log.append({
    "tool": "calculator",
    "arguments": {
        "a": 25,
        "b": 40,
        "operation": "multiply"
    },
    "status": "success"
})

print(tool_log)

[{'tool': 'calculator', 'arguments': {'a': 25, 'b': 40, 'operation': 'multiply'}, 'status': 'success'}]


# Why Log Tool Calls?

Logging helps developers understand:

- Which tool was called
- What arguments were provided
- Whether execution succeeded
- How long it took
- What error occurred

This is extremely useful when debugging agents.

When an agent does something strange, the logs can answer:

"What on earth did it actually call?"

In [18]:
calculator_tool = {
    "name": "calculator",
    "description": (
        "Perform basic arithmetic operations. "
        "Use this tool when an exact numerical calculation "
        "is required."
    ),
    "parameters": {
        "type": "object",
        "properties": {
            "a": {
                "type": "number",
                "description": "First number"
            },
            "b": {
                "type": "number",
                "description": "Second number"
            },
            "operation": {
                "type": "string",
                "enum": [
                    "add",
                    "subtract",
                    "multiply",
                    "divide"
                ],
                "description": "Arithmetic operation"
            }
        },
        "required": [
            "a",
            "b",
            "operation"
        ]
    }
}

print(calculator_tool)

{'name': 'calculator', 'description': 'Perform basic arithmetic operations. Use this tool when an exact numerical calculation is required.', 'parameters': {'type': 'object', 'properties': {'a': {'type': 'number', 'description': 'First number'}, 'b': {'type': 'number', 'description': 'Second number'}, 'operation': {'type': 'string', 'enum': ['add', 'subtract', 'multiply', 'divide'], 'description': 'Arithmetic operation'}}, 'required': ['a', 'b', 'operation']}}


# 24. Constrained Tool Arguments

Suppose a calculator accepts:

operation

Possible values:

- add
- subtract
- multiply
- divide

Instead of allowing any text, we can define an allowed set.

This is called an enum.

It reduces invalid tool calls.

Example:

"multiply"

Valid.

"do something"

Invalid.

Structured constraints make tool calling more reliable.

# 25. Connecting Tools to an LLM

A typical LLM API request can include:

1. User message
2. Available tool definitions

Conceptually:

LLM Request
{
    "messages": [...],
    "tools": [
        calculator_tool,
        search_tool,
        database_tool
    ]
}

The model can then decide whether to:

- Answer directly
- Request a tool call

# Example Tool Call Response

User:

"What is 25 × 40?"

The model may return structured information such as:

{
    "tool_name": "calculator",
    "arguments": {
        "a": 25,
        "b": 40,
        "operation": "multiply"
    }
}

The application then executes:

calculator(
    25,
    40,
    "multiply"
)

The result:

1000

The application sends that result back to the model.

The model can then produce the final answer.

# 26. Function Calling vs Normal Prompting

## Normal Prompt

You might tell an LLM:

"Calculate 25 × 40."

The model generates:

"1000"

The model itself generated the result.

---

## Function Calling

You provide:

Calculator tool

The model requests:

calculator(
    25,
    40,
    "multiply"
)

The application performs the calculation.

This is preferable when exact external execution is required.

# 27. Why Function Calling Matters

Function calling allows LLM applications to connect language
understanding with real software capabilities.

Without tools:

LLM
↓
Text

With tools:

LLM
↓
Decision
↓
Software Function
↓
Real Result
↓
LLM
↓
Text

This allows AI systems to interact with:

- Databases
- APIs
- Files
- Search systems
- Business applications
- Internal services

# 28. RAG as a Tool

Remember Week 8?

We built a RAG pipeline.

That RAG system can become an agent tool.

Example:

Tool:

search_company_documents(question)

The agent can decide:

"I need information from the company's documents."

Then:

Agent
 ↓
RAG Tool
 ↓
Vector Database
 ↓
Relevant Chunks
 ↓
Tool Result
 ↓
Agent
 ↓
Final Answer

This is how the Week 8 RAG system connects to Week 9 Agents.

In [19]:
def search_company_documents(question):
    return {
        "source": "employee_handbook.pdf",
        "page": 3,
        "content": (
            "Employees may work remotely up to two days "
            "per week with manager approval."
        )
    }


result = search_company_documents(
    "What is the remote work policy?"
)

print(result)

{'source': 'employee_handbook.pdf', 'page': 3, 'content': 'Employees may work remotely up to two days per week with manager approval.'}


# 29. Example Agent Toolset

Our future Multi-Agent Assistant could contain:

## Research Agent

Tools:
- Web Search
- RAG Search

## Data Agent

Tools:
- Python
- Database
- Calculator
- CSV Reader

## Writer Agent

Tools:
- Document Generator
- Formatting Tool

## Main Agent

Tools:
- Research Agent
- Data Agent
- Writer Agent

This gives each component a clear responsibility.

# 30. Complete Architecture
```
                         USER
                           ↓
                       MAIN AGENT
                           ↓
                    ┌──────┴──────┐
                    ↓             ↓
                 DECIDE       DIRECT ANSWER
                    ↓
               TOOL REQUIRED?
                    ↓
                   YES
                    ↓
              SELECT TOOL
                    ↓
             GENERATE ARGS
                    ↓
              VALIDATE ARGS
                    ↓
              EXECUTE TOOL
                    ↓
              TOOL RESULT
                    ↓
                OBSERVE
                    ↓
             TASK COMPLETE?
               ↙         ↘
             NO          YES
             ↓             ↓
        ANOTHER TOOL     ANSWER
             ↓
           LOOP

# 31. Exercise 1 - Create a Tool

Create a Python function:

convert_temperature(celsius)

It should convert Celsius to Fahrenheit.

Formula:

F = (C × 9/5) + 32

Test it with:

25°C

Expected result:

77°F

# Exercise 2 - Tool Definition

Create a tool definition for:

get_employee(employee_id)

The tool should:

- Accept employee_id
- Return employee information
- Clearly describe the tool
- Define the parameter type
- Mark employee_id as required

# Exercise 3 - Tool Selection

You have these tools:

1. calculator
2. search_web
3. read_file
4. query_database

Choose the correct tool for each request.

### A
"Calculate 450 × 25."

### B
"Find information about the latest AI models."

### C
"Read the contents of report.pdf."

### D
"Find employee E101."

Explain why you selected each tool.

# Exercise 4 - Design a Tool Schema

Design a tool called:

get_order_details(order_id)

Include:

- Name
- Description
- Parameter
- Parameter type
- Required field

Then create it as a Python dictionary.

# Exercise 5 - Agent Tool Flow

Design the function-calling workflow for:

"Find the total sales for order 1024."

Available tools:

- get_order_details(order_id)
- calculator(a, b, operation)

Your workflow should explain:

1. User request
2. First tool
3. Tool arguments
4. First result
5. Second tool
6. Second result
7. Final answer

# 32. Interview Questions and Answers

### Q1. What is function calling?

Function calling is a mechanism that allows an LLM to request
execution of predefined functions or tools using structured
arguments.

---

### Q2. Does the LLM directly execute the function?

Usually, no.

The LLM generates a structured tool-call request.
The application validates and executes the function.

---

### Q3. What is a tool schema?

A tool schema describes the tool's name, purpose, parameters,
types, and required fields so that the model can use it correctly.

---

### Q4. Why are tool descriptions important?

The model uses tool descriptions to understand when a tool is
appropriate and what it can do.

---

### Q5. What is tool selection?

Tool selection is the process of determining which available
tool should be used for the user's request.

---

### Q6. What are tool arguments?

Tool arguments are the input values required by the selected tool.

Example:

get_weather(
    city="Chennai"
)

Here, "Chennai" is a tool argument.

---

### Q7. Why should tool arguments be validated?

Validation prevents malformed or unsafe inputs from reaching
the underlying function or external system.

---

### Q8. What happens when a tool fails?

The application should catch the error and return a structured
result so that the agent can decide how to continue.

---

### Q9. What is a tool registry?

A tool registry maps tool names to their corresponding functions
or implementations.

---

### Q10. What is sequential tool calling?

Sequential tool calling means one tool is executed after another,
where later actions may depend on earlier results.

---

### Q11. What is parallel tool calling?

Parallel tool calling means multiple independent tools or tool
calls can be executed concurrently when the system supports it
and the operations are independent.

---

### Q12. Why are write tools more sensitive than read tools?

Write tools can modify external systems and create real-world
side effects.

Examples:

- Sending emails
- Updating databases
- Deleting files

They may therefore require stronger authorization or user approval.

---

### Q13. What is least privilege in agent systems?

An agent should receive only the tools and permissions required
for its task.

---

### Q14. Can RAG be a tool?

Yes.

A RAG search function can be exposed as a tool that an agent
calls when it needs information from a document knowledge base.

---

### Q15. Why is function calling useful for AI Agents?

It connects the LLM's decision-making capabilities with real
software actions and external systems.

# 33. Day 2 Cheat Sheet

## Tool

An external function or capability available to an agent.

## Function

A piece of code that accepts inputs and returns an output.

## Function Calling

LLM requests execution of a predefined function using structured
arguments.

## Tool Schema

Description of:

- Tool name
- Description
- Parameters
- Parameter types
- Required fields

## Tool Selection

Choosing the appropriate tool for a task.

## Tool Arguments

Values supplied to the selected tool.

## Tool Execution

Application executes the actual function.

## Tool Result

Output returned by the function.

## Tool Registry

Mapping between tool names and tool implementations.

## Validation

Checking tool arguments before execution.

## Error Handling

Managing failures during tool execution.

## Tool Permissions

Controlling which tools an agent can use.

## Least Privilege

Give an agent only the capabilities it needs.

## Read Tool

Retrieves information.

## Write Tool

Changes external state.

## Function Calling Flow

User
↓
LLM
↓
Tool Selection
↓
Arguments
↓
Validation
↓
Tool Execution
↓
Tool Result
↓
LLM
↓
Final Answer

# 35. Day 2 Final Summary

Today we learned how AI Agents interact with external tools.

## Basic Function

Function
↓
Input
↓
Processing
↓
Output

## AI Tool

Tool
↓
Description
↓
Parameters
↓
Function
↓
Result

## Function Calling

User
↓
LLM
↓
Choose Tool
↓
Generate Arguments
↓
Application
↓
Validate
↓
Execute Function
↓
Tool Result
↓
LLM
↓
Final Answer

## Multiple Tools

                    AGENT
                      ↓
             ┌────────┼────────┐
             ↓        ↓        ↓
        Calculator  Search  Database
             │        │        │
             └────────┼────────┘
                      ↓
                   Results
                      ↓
                  LLM / Agent
                      ↓
                  Final Answer

## Important Principle

The LLM decides what tool should be used.

The application executes the tool.

The tool returns a result.

The LLM interprets that result.

## Security Principle

Never give an agent more access than it needs.

Use:

- Validation
- Authorization
- Least privilege
- Logging
- Rate limits
- Approval for sensitive actions